# Clustering d'Images avec Scikit-learn et Visualisation avec Streamlit

Le clustering est une technique d'apprentissage non supervisé qui permet de regrouper des données similaires en différents groupes ou clusters. Dans le contexte de l'analyse d'images, le clustering peut être utilisé pour segmenter des images, détecter des objets ou des régions d'intérêt, et comprendre la structure des données d'image. Ce TP vous guidera à travers le processus de clustering d'un ensemble d'images de chiffres manuscrits en utilisant l'algorithme K-Means et l'extraction de caractéristiques HOG et d'histogrammes de niveaux de gris. Vous visualiserez ensuite les résultats du clustering à l'aide de la bibliothèque Streamlit.


À la fin de ce TP, vous aurez une compréhension pratique du processus de clustering d'images, de l'extraction de caractéristiques, de l'évaluation des performances et de la visualisation des résultats à l'aide de Streamlit. Ces compétences sont précieuses dans de nombreux domaines tels que la vision par ordinateur, le traitement d'images médicales, l'analyse de données d'imagerie satellitaire, et bien d'autres.


In [ ]:
from sklearn.preprocessing import StandardScaler
import os
import pandas as pd
from sklearn import datasets

from features import *
from clustering import *
from utils import *
from constant import  PATH_OUTPUT, MODEL_CLUSTERING



## Partie 1 : Création du modèle de clustering d'images
#### (fichier pipeline.py)

**1. Chargement des données d'images de chiffres manuscrits à partir du dataset Digits.**
   - Vous utiliserez le célèbre dataset Digitis qui contient des images de chiffres manuscrits. Ce dataset est souvent utilisé pour tester des algorithmes de reconnaissance de chiffres et d'apprentissage automatique.



In [2]:
print("##### Chargement des données ######")
digits = datasets.load_digits()
labels_true =digits.target
images = digits.images


##### Chargement des données ######


**2. Extraction des caractéristiques HOG (Histogrammes de Gradients Orientés) et des histogrammes de niveaux de gris à partir des images.**
   - Les caractéristiques HOG capturent les informations de gradient et de bords dans les images, ce qui les rend utiles pour la reconnaissance d'objets et de formes.
   - Les histogrammes de niveaux de gris représentent la distribution des intensités de pixels dans l'image, fournissant des informations sur la texture et les motifs.

**TODO :**
   - Implémentez les fonctions `compute_hog_descriptors` et `compute_gray_histograms` dans  le fichier `features.py`, utilisez respectivement les fonctions `hog` de  la librairie `skimage` et  `calcHist` de `cv2`.
   - lien HOG : https://scikit-image.org/docs/stable/auto_examples/features_detection/plot_hog.html
   - lien  histogrammes de niveaux de gris : https://pyimagesearch.com/2021/04/28/opencv-image-histograms-cv2-calchist/
   

In [3]:
print("\n\n ##### Extraction de Features ######")
print("- calcul features hog...")
descriptors_hog = compute_hog_descriptors(images)
print("- calcul features Histogram...")
descriptors_hist = compute_gray_histograms(images)
descriptors_hist
descriptors_hog




 ##### Extraction de Features ######
- calcul features hog...
- calcul features Histogram...


array([[0.5       , 0.5       , 0.5       , ..., 0.        , 0.        ,
        0.        ],
       [0.53062065, 0.50149743, 0.53062065, ..., 0.        , 0.        ,
        0.70710678],
       [0.52262309, 0.52262309, 0.05950701, ..., 0.40629917, 0.40629917,
        0.40629917],
       ...,
       [0.50876588, 0.        , 0.50876588, ..., 0.05019662, 0.5729694 ,
        0.        ],
       [0.40930724, 0.40930724, 0.40930724, ..., 0.        , 0.53195353,
        0.        ],
       [0.52160591, 0.42869782, 0.        , ..., 0.        , 0.52256362,
        0.        ]])


**3. Application de l'algorithme K-Means sur les caractéristiques extraites pour obtenir les clusters.**
   - L'algorithme K-Means est un algorithme de clustering populaire qui partitionne les données en K clusters en minimisant la somme des carrés des distances entre les points de données et les centroïdes des clusters.
   
   
 
 **TODO :**
   - Dans le fichier `clustering.py` implémentez les fonctions `initialize_centers()`, `nearest_cluster()` et `fit()` du KMeans.
   

In [4]:
print("\n\n ##### Clustering ######")
number_cluster = 10
kmeans_hog = KMeans(n_clusters=number_cluster)
kmeans_hist = KMeans(n_clusters=number_cluster)

print("- calcul kmeans avec features HOG ...")
kmeans_hog.fit(np.array(descriptors_hog))
print("- calcul kmeans avec features Histogram...")
kmeans_hist.fit(np.array(descriptors_hist))



 ##### Clustering ######
- calcul kmeans avec features HOG ...
- calcul kmeans avec features Histogram...


**4. Évaluation des performances du clustering en utilisant des métriques telles que la pureté, l'entropie, et les scores Rand et Mutual Information.**
   - La pureté mesure la fraction d'exemples de cluster qui sont membres du cluster majoritaire.
   - L'entropie est une mesure de désordre ou d'impureté dans les clusters.
   - Le score Rand mesure la similarité entre deux partitions en comparant les paires d'exemples.
   - Le score Mutual Information évalue la quantité d'information partagée entre deux partitions.


In [5]:

print("\n\n##### Résultat ######")
metric_hist = show_metric(labels_true, kmeans_hist.labels_, descriptors_hist, bool_show=True, name_descriptor="HISTOGRAM", bool_return=True)
print("\n\n")
metric_hog = show_metric(labels_true, kmeans_hog.labels_, descriptors_hog,bool_show=True, name_descriptor="HOG", bool_return=True)




##### Résultat ######
########## Métrique descripteur : HISTOGRAM
Adjusted Rand Index: 0.06438753701884177
Jaccard Index: 0.0378930476154567
Homogeneity: 0.13538065890821213
Completeness: 0.13810369152330118
V-measure: 0.13672861885209062
Silhouette Score: 0.06385773420333862
Adjusted Mutual Information: 0.12799681125936896



########## Métrique descripteur : HOG
Adjusted Rand Index: 0.38730872477517453
Jaccard Index: 0.06469914718311216
Homogeneity: 0.507424295173278
Completeness: 0.517391823349658
V-measure: 0.5123595863617199
Silhouette Score: 0.061052269499643874
Adjusted Mutual Information: 0.5074268721041726


**5. Conversion des données de clustering au format requis pour la visualisation avec Streamlit.**


In [6]:
list_dict = [metric_hist,metric_hog]
df_metric = pd.DataFrame(list_dict)

scaler = StandardScaler()
descriptors_hist_norm = scaler.fit_transform(descriptors_hist)
descriptors_hog_norm = scaler.fit_transform(descriptors_hog)

#conversion vers un format 3D pour la visualisation
x_3d_hist = conversion_3d(descriptors_hist_norm)
x_3d_hog = conversion_3d(descriptors_hog_norm)

# création des dataframe pour la sauvegarde des données pour la visualisation
df_hist = create_df_to_export(x_3d_hist, labels_true, kmeans_hist.labels_)
df_hog = create_df_to_export(x_3d_hog, labels_true, kmeans_hog.labels_)

# Vérifie si le dossier existe déjà
if not os.path.exists(PATH_OUTPUT):
    # Crée le dossier
    os.makedirs(PATH_OUTPUT)

# sauvegarde des données
df_hist.to_excel(PATH_OUTPUT+"/save_clustering_hist_kmeans.xlsx")
df_hog.to_excel(PATH_OUTPUT+"/save_clustering_hog_kmeans.xlsx")
df_metric.to_excel(PATH_OUTPUT+"/save_metric.xlsx")

***6. Création du fichier pipeline.py*** 
- Mettez au propre le code dans le fichier pipeline.py
- Puis exécutez :  `python pipeline.py `


## Partie 2 : Visualisation des résultats du clustering avec Streamlit
### (fichier dashboad_clustering.py)

Cette partie constituera le rendu final du TP. Nous développerons une application Streamlit pour visualiser et analyser les résultats du clustering.

L'application permettra de :

***1. Visualisation 3D du clustering***
- Nous créerons une visualisation 3D interactive des clusters obtenus, avec la possibilité de mettre en évidence un cluster spécifique et d'afficher des exemples d'images appartenant à ce cluster.

***TODO :***
- Utilizer la fonction `scatter_3d()` pour faire un plot 3D du clustering.
- lien : https://plotly.com/python/3d-scatter-plots/




In [7]:
import matplotlib.pyplot as plt
import plotly.express as px

In [8]:
fig = px.scatter_3d(df_hist, x='x', y='y', z='z', color='cluster')


***2. Métriques d'évaluation***
- Nous calculerons et afficherons diverses métriques d'évaluation, telles que le score AMI (Adjusted Mutual Information), pour quantifier la qualité du clustering obtenu avec chaque descripteur.

***TODO :*** 
- Utilisez la fonction `px.bar()` pour afficher un histogramme du score AMI.
- lien : https://plotly.com/python/horizontal-bar-charts/

In [9]:
graph_size = 300


In [10]:


fig = px.bar(df_metric, x="descriptor", y="ami", color="descriptor",
             title="Score AMI par descripteur")
fig

C:\Users\preve\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\plotly\express\_core.py:1979: FutureWarning:

When grouping with a length-1 list-like, you will need to pass a length-1 tuple to get_group in a future version of pandas. Pass `(name,)` instead of `name` to silence this warning.



***3. Finalisation du fichier dashboard_clustering.py***

***TODO :***
- Ajoutez les graphiques dans le fichier, puis lancez la commande :  `streamlit run dashboard_clustering.py `